In [1]:
!pip install pandas openpyxl sqlalchemy pymysql


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

df = pd.read_excel("fifa_players_mod.xlsx")

players = df.copy()

players.columns = (
    players.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("(1-5)", "", regex=False)
)

players["value_euro"] = players["value_euro"].fillna(0)
players["national_team"] = players["national_team"].fillna("None")
players["national_team_position"] = players["national_team_position"].fillna("None")
players["national_jersey_number"] = players["national_jersey_number"].fillna(0)
players.drop_duplicates(inplace=True)

players["primary_position"] = players["positions"].str.split(",").str[0]
players["is_international"] = (players["national_team"] != "None").astype(int)

def age_group(age):
    if age <= 21:
        return "Youth (<=21)"
    elif age <= 29:
        return "Prime (22-29)"
    return "Veteran (30+)"

players["age_group"] = players["age"].apply(age_group)

skill_cols = ["crossing", "finishing", "heading_accuracy", "dribbling", "curve",
              "freekick_accuracy", "long_passing", "sprint_speed", "shot_power",
              "jumping", "stamina", "strength", "penalties"]
players["total_skill_score"] = players[skill_cols].sum(axis=1)
players["value_per_rating"] = (players["value_euro"] / players["overall_rating"]).round(2)

print("Original Shape:", df.shape)
print("Cleaned Shape:", players.shape)
players.head()


Original Shape: (17954, 26)
Cleaned Shape: (17954, 31)


,name,full_name,age,height_cm,positions,nationality,overall_rating,value_euro,preferred_foot,weak_foot,...,shot_power,jumping,stamina,strength,penalties,primary_position,is_international,age_group,total_skill_score,value_per_rating
0,L. Messi,Lionel Andrés Messi Cuccittini,31,170.18,"CF,RW,ST",Argentina,94,110500000.0,Left,4,...,85,68,72,66,75,CF,1,Veteran (30+),1076,1175531.91
1,C. Eriksen,Christian Dannemann Eriksen,27,154.94,"CAM,RM,CM",Denmark,88,69500000.0,Right,5,...,84,50,92,58,67,CAM,1,Prime (22-29),991,789772.73
2,P. Pogba,Paul Pogba,25,190.50,"CM,CAM",France,88,73000000.0,Right,4,...,90,83,88,87,82,CM,1,Prime (22-29),1083,829545.45
3,L. Insigne,Lorenzo Insigne,27,162.56,"LW,ST",Italy,88,62000000.0,Right,4,...,75,53,75,44,61,LW,1,Prime (22-29),945,704545.45
4,K. Koulibaly,Kalidou Koulibaly,27,187.96,CB,Senegal,88,60000000.0,Right,3,...,55,81,75,94,33,CB,0,Prime (22-29),733,681818.18


In [5]:
from sqlalchemy import create_engine, text
import urllib.parse

username = "root"
password = "Yuvaraj@18"
safe_password = urllib.parse.quote_plus(password)
host = "localhost"
port = 3306


In [6]:
server_engine = create_engine(f"mysql+pymysql://{username}:{safe_password}@{host}:{port}")

with server_engine.connect() as connection:
    connection.execute(text("CREATE DATABASE IF NOT EXISTS player_analytics"))

print("Database ready: player_analytics")


Database ready: player_analytics


In [7]:
db_engine = create_engine(f"mysql+pymysql://{username}:{safe_password}@{host}:{port}/player_analytics")
print("Connected to player_analytics!")


Connected to player_analytics!


In [8]:
players.to_sql(
    "fifa_players",
    con=db_engine,
    if_exists="replace",
    index=False,
    chunksize=5000
)
print("Data loaded successfully!")


Data loaded successfully!


In [9]:
pd.read_sql("SELECT COUNT(*) AS total_rows FROM fifa_players", db_engine)

,total_rows
0,17954


In [10]:
pd.read_sql("SELECT * FROM fifa_players LIMIT 5", db_engine)

,name,full_name,age,height_cm,positions,nationality,overall_rating,value_euro,preferred_foot,weak_foot,...,shot_power,jumping,stamina,strength,penalties,primary_position,is_international,age_group,total_skill_score,value_per_rating
0,L. Messi,Lionel Andrés Messi Cuccittini,31,170.18,"CF,RW,ST",Argentina,94,110500000.0,Left,4,...,85,68,72,66,75,CF,1,Veteran (30+),1076,1175531.91
1,C. Eriksen,Christian Dannemann Eriksen,27,154.94,"CAM,RM,CM",Denmark,88,69500000.0,Right,5,...,84,50,92,58,67,CAM,1,Prime (22-29),991,789772.73
2,P. Pogba,Paul Pogba,25,190.50,"CM,CAM",France,88,73000000.0,Right,4,...,90,83,88,87,82,CM,1,Prime (22-29),1083,829545.45
3,L. Insigne,Lorenzo Insigne,27,162.56,"LW,ST",Italy,88,62000000.0,Right,4,...,75,53,75,44,61,LW,1,Prime (22-29),945,704545.45
4,K. Koulibaly,Kalidou Koulibaly,27,187.96,CB,Senegal,88,60000000.0,Right,3,...,55,81,75,94,33,CB,0,Prime (22-29),733,681818.18


In [11]:
total_players = len(players)
total_market_value = players["value_euro"].sum()
avg_value = players["value_euro"].mean()
avg_rating = players["overall_rating"].mean()
intl_rate = players["is_international"].mean() * 100

print(f"Total Players: {total_players:,}")
print(f"Total Market Value: EUR {total_market_value:,.0f}")
print(f"Average Market Value: EUR {avg_value:,.0f}")
print(f"Average Overall Rating: {avg_rating:.2f}")
print(f"International Call-up Rate: {intl_rate:.2f}%")


Total Players: 17,954
Total Market Value: EUR 43,880,780,000
Average Market Value: EUR 2,444,067
Average Overall Rating: 66.24
International Call-up Rate: 4.77%


In [12]:
print("Top 5 nationalities by player count:")
print(players["nationality"].value_counts().head())

print()
print("Top 5 most valuable players:")
print(players.nlargest(5, "value_euro")[["name","nationality","primary_position","overall_rating","value_euro"]])

print()
print("Average market value by age group:")
print(players.groupby("age_group")["value_euro"].mean().round(0))


Top 5 nationalities by player count:
nationality
England      1658
Germany      1199
Spain        1070
France        925
Argentina     904
Name: count, dtype: int64

Top 5 most valuable players:
               name nationality primary_position  overall_rating   value_euro
0          L. Messi   Argentina               CF              94  110500000.0
17943     Neymar Jr      Brazil               LW              92  108000000.0
17941  K. De Bruyne     Belgium              CAM              91  102000000.0
17937       H. Kane     England               ST              90   96500000.0
17940     E. Hazard     Belgium               LW              91   93000000.0

Average market value by age group:
age_group
Prime (22-29)    2990275.0
Veteran (30+)    2486255.0
Youth (<=21)     1086128.0
Name: value_euro, dtype: float64
